# Data exploration of MTSamples

General exploration of the MTSamples transcription records, looking for patterns in how each note is structured, and specifically how the diagnosis is presented/displayed in the `transcription` note across the specialties or document type. 

In [1]:
# Libraries
import pandas as pd

In [2]:
pd.set_option("display.max_colwidth", None)

In [3]:
# Load the dataset 
df = pd.read_csv("../data/mtsamples.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4999 entries, 0 to 4998
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Unnamed: 0         4999 non-null   int64
 1   description        4999 non-null   str  
 2   medical_specialty  4999 non-null   str  
 3   sample_name        4999 non-null   str  
 4   transcription      4966 non-null   str  
 5   keywords           3931 non-null   str  
dtypes: int64(1), str(5)
memory usage: 234.5 KB


Total rows: 4999
Total columns: 6  
Data types: ints, str  
Missing values: 33 missing transcripts, 1,068 missing keywords  
Index column: Unnamed

In [4]:
# First 5 records
df.head()

Unnamed: 0  \
0           0   
1           1   
2           2   
3           3   
4           4   

                                                         description  \
0   A 23-year-old white female presents with complaint of allergies.   
1                           Consult for laparoscopic gastric bypass.   
2                           Consult for laparoscopic gastric bypass.   
3                                             2-D M-Mode. Doppler.     
4                                                 2-D Echocardiogram   

             medical_specialty                                sample_name  \
0         Allergy / Immunology                         Allergic Rhinitis    
1                   Bariatrics   Laparoscopic Gastric Bypass Consult - 2    
2                   Bariatrics   Laparoscopic Gastric Bypass Consult - 1    
3   Cardiovascular / Pulmonary                    2-D Echocardiogram - 1    
4   Cardiovascular / Pulmonary                    2-D Echocardiogram - 2    

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            

In [5]:
# Missing values 
df.isna().sum()

Unnamed: 0              0
description             0
medical_specialty       0
sample_name             0
transcription          33
keywords             1068
dtype: int64

33 transcriptions missing values

In [6]:
# Number of unique values under `medical_specialty` field.
df["medical_specialty"].nunique()

40

In [7]:
# Unique values under `medical_specialty` field.
df["medical_specialty"].value_counts()

medical_specialty
Surgery                          1103
Consult - History and Phy.        516
Cardiovascular / Pulmonary        372
Orthopedic                        355
Radiology                         273
General Medicine                  259
Gastroenterology                  230
Neurology                         223
SOAP / Chart / Progress Notes     166
Obstetrics / Gynecology           160
Urology                           158
Discharge Summary                 108
ENT - Otolaryngology               98
Neurosurgery                       94
Hematology - Oncology              90
Ophthalmology                      83
Nephrology                         81
Emergency Room Reports             75
Pediatrics - Neonatal              70
Pain Management                    62
Psychiatry / Psychology            53
Office Notes                       51
Podiatry                           47
Dermatology                        29
Dentistry                          27
Cosmetic / Plastic Surgery      

In [8]:
# Select General Medicine to get an instinct of the transcript structure
gen_med = df[df["medical_specialty"].str.strip() == "General Medicine"]
gen_med.shape

(259, 6)

Had to strip the field names of trailing whitespaces

In [9]:
# Inspect a transcription from a gen_med subset dataset
print(gen_med["transcription"].iloc[1])

SUBJECTIVE:  ,This 68-year-old man presents to the emergency department for three days of cough, claims that he has brought up some green and grayish sputum.  He says he does not feel short of breath.  He denies any fever or chills.,REVIEW OF SYSTEMS:,HEENT:  Denies any severe headache or sore throat.,CHEST:  No true pain.,GI:  No nausea, vomiting, or diarrhea.,PAST HISTORY:,  He states that he is on Coumadin because he had a cardioversion done two months ago for atrial fibrillation.  He also lists some other medications.  I do have his medications list.  He is on Pacerone, Zaroxolyn, albuterol inhaler, Neurontin, Lasix, and several other medicines.  Those are the predominant medicines.  He is not a diabetic.  The past history otherwise, he has had smoking history, but he quit several years ago and denies any COPD or emphysema.  No one else in the family is sick.,PHYSICAL EXAMINATION:,GENERAL:  The patient appears comfortable.  He did not appear to be in any respiratory distress.  He w

There may be data leakage under the following sections IMPRESSION, PLAN, ASSESSMENT, and DIAGNOSIS. 

In [10]:
# Percentage of specialty transcriptions that contain DIAGNOSIS, IMPRESSION, ASSESSMENT, PLAN sections.
header_coverage = df["transcription"].str.contains(
    "DIAGNOSIS|IMPRESSION|ASSESSMENT|PLAN", case=False, na=False
)

coverage_by_specialty = header_coverage.groupby(df["medical_specialty"]).mean()
coverage_by_specialty.sort_values(ascending=False)

medical_specialty
Allergy / Immunology             1.000000
Diets and Nutritions             1.000000
Physical Medicine - Rehab        0.952381
Endocrinology                    0.947368
Nephrology                       0.913580
Psychiatry / Psychology          0.905660
Rheumatology                     0.900000
Bariatrics                       0.888889
Dentistry                        0.888889
Lab Medicine - Pathology         0.875000
Cosmetic / Plastic Surgery       0.851852
Neurology                        0.843049
Hospice - Palliative Care        0.833333
SOAP / Chart / Progress Notes    0.831325
Podiatry                         0.829787
Emergency Room Reports           0.826667
Orthopedic                       0.822535
Radiology                        0.820513
Consult - History and Phy.       0.817829
Pediatrics - Neonatal            0.800000
Urology                          0.797468
Dermatology                      0.793103
Gastroenterology                 0.778261
Hematology - Onc

In [13]:
# Checking transcriptions that do not contain PLAN, DIAGNOSIS, ASSESSMENT, and IMPRESSION
misses = df[~header_coverage] 
for _, row in misses.sample(5, random_state=1).iterrows():
    print(f"=== {row['medical_specialty'].strip()} | {row['sample_name']} ===")
    print(row["transcription"])
    print()

=== Pain Management |  Depo-Medrol Injection  ===
PROCEDURE: , Right L5-S1 intralaminar epidural steroid injection with 120 mg of Depo-Medrol under fluoroscopic guidance.,INDICATION: , The patient is a 51-year-old female with back pain referring into the right leg.,RISKS VERSUS BENEFITS: , The risks and benefits were discussed with the patient prior to the procedure.  She agrees to accept the risks and signs a written consent to proceed with the procedure.,DESCRIPTION OF PROCEDURE: , The patient was placed prone on the table.  The skin was thoroughly cleansed with Betadine swabs x3 and wiped off with a sterile gauze.  The subcutaneous intramuscular and interligamentous region was anesthetized with 4% lidocaine.,A 3-1/2-inch 20-gauge Tuohy catheter was directed under intermittent fluoroscopic guidance at the lamina.  Once the lamina was detected, the catheter was directed cephalad and medially and loss of resistance technique was used to determine the epidural space.,EPIDUROGRAM: , Omni

## Summary & Next Steps

[MTSamples](https://www.mtsamples.com/index.asp) is a collection of clinical transcriptions across specialties, with identifying details altered. This dataset was acquired from Kaggle via the `kagglehub` API. 4,999 records, 6 columns, 33 missing transcriptions, 1,068 missing keywords. Of the 40 distinct `medical_specialty` values, several aren't specialties at all but document types (e.g. Discharge Summary, SOAP/Chart/Progress Note).

The task under exploration was to have the LLM (MedGemma 1.5:4B) generate a differential diagnosis given a transcription after the section containing the diagnosis (DIAGNOSIS/IMPRESSION/PLAN/ASSESSMENT) had been stripped from the transcription. Header coverage varies widely by specialty, from 33% (Office Notes) to 100% (Allergy/Immunology), and sits around 73% even for General Medicine. Reviewing a sample of the misses found no dominant pattern. Some notes weave the diagnosis through the rest of the note's text with no section to remove at all. 

Diagnosis leakage in transcription can't be reliably prevented by header-based stripping on this dataset. 

I have decided to move away from MTSamples for this task rather than continuing to expand the stripping heuristic. I performed an online dataset search and will start to evaluate `starmpcc/Asclepius-Synthetic-Clinical-Notes` (available at Hugging Face) as a replacement.